# ⚽ Análisis y Predicción de Partidos de Fútbol

Este notebook sirve como guía interactiva para entender el funcionamiento del **Modelo Predictivo de Fútbol** desarrollado en el proyecto. Veremos cómo interactuar con los ratings ELO y cómo estimar los goles esperados (xG) y las probabilidades de partido usando la Distribución de Poisson.

### 1. Configuración de importaciones y variables del sistema
Importamos los módulos locales y la librería Pandas para analizar los datos.

In [ ]:
import sys
import os
#Estas importaciones de arriba son propias de python y nos permitirán abajo trabajar con distintos directorios
import pandas as pd

# Añadimos el directorio raíz del proyecto al sys.path para poder importar src correctamente
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.utils import obtener_conexion
from src.elo import obtener_elo_equipo
from src.poisson import predecir_probabilidades_goles, obtener_probabilidades_1X2

### 2. Exploración de los Ratings ELO de las Selecciones
Cargamos la tabla de ELO desde la base de datos SQLite y visualizamos el Top 10 de las selecciones con mayor rating calculado.

In [ ]:
#Consultas sql
conexion = obtener_conexion()
df_elo = pd.read_sql_query("""
    SELECT team AS Selección, rating AS [Rating ELO], confederation AS Confederación, fifa_rank AS [Ranking FIFA]
    FROM ratings_elo
    ORDER BY rating DESC
    LIMIT 10
""", conexion)
conexion.close()

df_elo

,Selección,Rating ELO,Confederación,Ranking FIFA
0,Italy,1685.52,Desconocido,NaN
1,France,1668.58,UEFA,1.0
2,Germany,1646.58,UEFA,10.0
3,Iran,1624.00,AFC,21.0
4,Netherlands,1605.39,UEFA,7.0
5,Egypt,1576.00,CAF,29.0
6,Brazil,1574.82,CONMEBOL,6.0
7,Canada,1570.00,CONCACAF,30.0
8,West Germany,1559.52,Desconocido,NaN
9,Ivory Coast,1552.00,CAF,33.0


### 3. Simulación de un Enfrentamiento Directo
Seleccionamos dos equipos y predecimos el resultado utilizando la distribución de Poisson ajustada por ELO.

In [3]:
equipo_A = "Argentina"
equipo_B = "France"

# Calculamos las tasas esperadas de goles (lambdas) y la matriz de probabilidad de goles exactos
xg_A, xg_B, matriz = predecir_probabilidades_goles(equipo_A, equipo_B, es_neutral=True)

# Obtenemos las probabilidades de victoria A, empate y victoria B
prob_A, prob_empate, prob_B = obtener_probabilidades_1X2(matriz)

print(f"=== Predicción de Partido Neutral ===")
print(f"{equipo_A} vs {equipo_B}\n")
print(f"Goles Esperados (xG) para {equipo_A}: {xg_A:.2f}")
print(f"Goles Esperados (xG) para {equipo_B}: {xg_B:.2f}\n")
print(f"Probabilidad de Victoria de {equipo_A}: {prob_A*100:.1f}%")
print(f"Probabilidad de Empate: {prob_empate*100:.1f}%")
print(f"Probabilidad de Victoria de {equipo_B}: {prob_B*100:.1f}%")

=== Predicción de Partido Neutral ===
Argentina vs France

Goles Esperados (xG) para Argentina: 0.70
Goles Esperados (xG) para France: 3.12

Probabilidad de Victoria de Argentina: 5.8%
Probabilidad de Empate: 11.4%
Probabilidad de Victoria de France: 82.7%
